# VideoMeta test: local Qwen window annotation

Run this only after `test_motion_spatial_lvlm_pipeline.ipynb` has prepared the window
artifacts with the updated joiner (`encode_frame_images=False`).

The earlier out-of-memory crash was not the video. It failed inside the prefill loop at
`mx.eval([c.state for c in prompt_cache])` because the prompt serialised all 150 frames of
each window as JSON: roughly 66,000 tokens, or about 10 GB of KV cache on its own for
Qwen3-VL-4B. The annotated MP4 itself is only ~1,100 visual tokens.

Use a fresh kernel so YOLO/MPS memory is released before Qwen loads.

In [18]:
# Run once, then restart the kernel.
# %pip install --quiet --upgrade -e "/Users/juanvargas/UTS/3rd Semester/Capstone/videometa" mlx-vlm

In [19]:
import json
import logging
from pathlib import Path

import mlx.core as mx

from videometa import LocalQwenEventAnnotator, PreparedWindowInput

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)

OUTPUT_DIR = Path("pipeline_output")
PREPARED_WINDOWS_PATH = OUTPUT_DIR / "prepared_windows.json"
ANNOTATIONS_PATH = OUTPUT_DIR / "window_event_annotations.json"
LOCAL_QWEN_MODEL = "mlx-community/Qwen3-VL-4B-Instruct-4bit"

# Cap the MLX buffer pool. By default it grows across loop iterations until an
# allocation fails mid-prefill.
mx.set_cache_limit(2 * 1024**3)

if not PREPARED_WINDOWS_PATH.is_file():
    raise FileNotFoundError(
        f"Missing {PREPARED_WINDOWS_PATH}. Run test_motion_spatial_lvlm_pipeline.ipynb first."
    )

## Load the prepared windows

In [20]:
prepared_manifest = json.loads(PREPARED_WINDOWS_PATH.read_text(encoding="utf-8"))
prepared_windows = []
for item in prepared_manifest:
    annotated_video_path = item.get("annotated_video_path")
    if not annotated_video_path or not Path(annotated_video_path).is_file():
        raise FileNotFoundError(
            f"Prepared window video is missing: {annotated_video_path}. "
            "Rerun the preparation notebook."
        )
    prepared_windows.append(
        PreparedWindowInput(
            start_seconds=item["start_seconds"],
            end_seconds=item["end_seconds"],
            object_features=tuple(item["object_features"]),
            frame_features=tuple(item["frame_features"]),
            image_messages=(),
            artifact_directory=item.get("artifact_directory"),
            annotated_video_path=annotated_video_path,
        )
    )

print(f"Loaded {len(prepared_windows)} prepared window videos")

Loaded 17 prepared window videos


## Sanity check before loading the model

Confirms the MP4s really are small and shows what the prompt will cost. If
`chars` here is in the hundreds of thousands, the joiner did not run with the
updated code and the annotation loop will crash again.

In [21]:
import subprocess

sample = prepared_windows[0]
print("Video:", sample.annotated_video_path)
try:
    probe = subprocess.run(
        ["ffprobe", "-v", "error", "-select_streams", "v:0",
         "-show_entries", "stream=width,height,nb_frames,avg_frame_rate",
         "-of", "default=noprint_wrappers=1", sample.annotated_video_path],
        capture_output=True, text=True, check=True,
    )
    print(probe.stdout)
except (FileNotFoundError, subprocess.CalledProcessError) as error:
    print(f"ffprobe unavailable ({error}); skipping probe")

raw = len(json.dumps(sample.frame_features))
print(f"Raw frame_features: {len(sample.frame_features)} frames, {raw:,} chars "
      f"(~{raw // 3:,} tokens if sent whole)")

Video: pipeline_output/window_frames/window_70.000s_82.000s/annotated_window.mp4
width=640
height=360
avg_frame_rate=4/1
nb_frames=32

Raw frame_features: 360 frames, 244,632 chars (~81,544 tokens if sent whole)


## Annotate every window

One model instance is reused. Each call clears the MLX cache afterwards, results are
written incrementally, and a failing window is recorded rather than aborting the run.

In [22]:
annotator = LocalQwenEventAnnotator(
    model_id=LOCAL_QWEN_MODEL,
    video_fps=2.0,             # must match QWEN_ANNOTATED_VIDEO_FPS in the pipeline notebook
    max_tokens=600,
    prompt_token_budget=6000,
    feature_frames=8,
)

events_by_window = []
failures = []

for index, prepared in enumerate(prepared_windows, start=1):
    label = f"{prepared.start_seconds:.1f}s-{prepared.end_seconds:.1f}s"
    print(f"Annotating window {index}/{len(prepared_windows)}: {label}")
    try:
        events = annotator.annotate(prepared)
    except Exception as error:
        print(f"  FAILED: {type(error).__name__}: {error}")
        failures.append({"window": label, "error": str(error)})
        mx.clear_cache()
        continue

    events_by_window.append({
        "start_seconds": prepared.start_seconds,
        "end_seconds": prepared.end_seconds,
        "annotated_video_path": prepared.annotated_video_path,
        "events": events,
    })
    print(f"  {len(events)} events | peak GPU memory: {mx.get_peak_memory() / 1024**3:.2f} GB")
    ANNOTATIONS_PATH.write_text(json.dumps(events_by_window, indent=2), encoding="utf-8")

print(f"\nSaved {len(events_by_window)} windows to {ANNOTATIONS_PATH.resolve()}")
if failures:
    print(f"{len(failures)} windows failed:")
    for failure in failures:
        print(" ", failure)

2026-08-31 17:23:21,392 | INFO | httpx | HTTP Request: GET https://huggingface.co/api/models/mlx-community/Qwen3-VL-4B-Instruct-4bit/revision/main "HTTP/1.1 200 OK"
Fetching 14 files: 100%|██████████| 14/14 [00:00<00:00, 2018.71it/s]
2026-08-31 17:23:22,743 | INFO | videometa.window_annotation | Loaded local Qwen model: mlx-community/Qwen3-VL-4B-Instruct-4bit
2026-08-31 17:23:22,749 | INFO | videometa.window_annotation | Annotating prepared window 70.000s-82.000s with local Qwen
2026-08-31 17:23:22,753 | INFO | videometa.window_annotation | Window prompt: 1041 tokens from 8 sampled frames


Annotating window 1/17: 70.0s-82.0s


2026-08-31 17:23:29,031 | INFO | videometa.window_annotation | Local Qwen returned 4 events for pipeline_output/window_frames/window_70.000s_82.000s/annotated_window.mp4
2026-08-31 17:23:29,035 | INFO | videometa.window_annotation | Annotating prepared window 80.000s-92.000s with local Qwen
2026-08-31 17:23:29,039 | INFO | videometa.window_annotation | Window prompt: 1395 tokens from 8 sampled frames


  4 events | peak GPU memory: 7.42 GB
Annotating window 2/17: 80.0s-92.0s


2026-08-31 17:23:32,834 | INFO | videometa.window_annotation | Local Qwen returned 2 events for pipeline_output/window_frames/window_80.000s_92.000s/annotated_window.mp4
2026-08-31 17:23:32,835 | INFO | videometa.window_annotation | Annotating prepared window 280.000s-292.000s with local Qwen
2026-08-31 17:23:32,838 | INFO | videometa.window_annotation | Window prompt: 1110 tokens from 8 sampled frames


...

  1 events | peak GPU memory: 7.62 GB
Annotating window 16/17: 90.0s-102.0s


2026-08-31 17:24:27,004 | INFO | videometa.window_annotation | Local Qwen returned 1 events for pipeline_output/window_frames/window_90.000s_102.000s/annotated_window.mp4
2026-08-31 17:24:27,005 | INFO | videometa.window_annotation | Annotating prepared window 130.000s-142.000s with local Qwen
2026-08-31 17:24:27,008 | INFO | videometa.window_annotation | Window prompt: 1327 tokens from 8 sampled frames


  1 events | peak GPU memory: 7.63 GB
Annotating window 17/17: 130.0s-142.0s


2026-08-31 17:24:29,491 | INFO | videometa.window_annotation | Local Qwen returned 1 events for pipeline_output/window_frames/window_130.000s_142.000s/annotated_window.mp4


  1 events | peak GPU memory: 7.63 GB

Saved 15 windows to /Users/juanvargas/UTS/3rd Semester/Capstone/videometa/tests/pipeline_output/window_event_annotations.json
2 windows failed:
  {'window': '290.0s-300.2s', 'error': 'Unterminated string starting at: line 1 column 2473 (char 2472)'}
  {'window': '110.0s-122.0s', 'error': "Expecting ',' delimiter: line 64 column 134 (char 2927)"}


## Inspect the results

In [24]:
print(json.dumps(events_by_window[:3], indent=2))

[
  {
    "start_seconds": 70.0,
    "end_seconds": 82.0,
    "annotated_video_path": "pipeline_output/window_frames/window_70.000s_82.000s/annotated_window.mp4",
    "events": [
      {
        "event_name": "Vehicle movement",
        "description": "A vehicle labeled as 'car #1' is seen moving from the right side of the frame towards the left, passing a designated parking area.",
        "involved_objects": [
          {
            "id": 1014,
            "label": "car",
            "physical_details": "A dark-colored vehicle moving from right to left across the bottom-right portion of the frame."
          }
        ]
      },
      {
        "event_name": "Vehicle movement",
        "description": "A vehicle labeled as 'car #721' is seen moving from the left side of the frame towards the right, passing a designated parking area.",
        "involved_objects": [
          {
            "id": 721,
            "label": "car",
            "physical_details": "A white vehicle moving fr